# Titanic 本番

**モード:** 本番（`competition/`）＝ **協力戦**。AI も道具として駆使して、実際にリーダーボードを上げにいく。
方向（何を試すか・いつ提出するか）は自分が決め、コードは Claude と一緒に書いて回す。

**現在地:** gender ベースライン提出済み → public **0.76555**（＝性別だけ見たときの天井）
**目標:** 自分のモデル＋特徴量で **0.76555 を超える**　／　**評価指標: accuracy（正解率）**

本番の流れ:
```
train.csv (Survived有) ──fit──▶ モデル ──predict──▶ test.csv (Survived無)
                                                        │
                                                        ▼
                                  submissions/submission.csv  (PassengerId, Survived)
                                                        │  kaggle competitions submit
                                                        ▼
                                                  リーダーボード
```
データ: `data/train.csv`(891) / `data/test.csv`(418) ／ 提出は `submissions/submission.csv`（2列・`index=False`）

## 1. 読み込み・EDA

In [1]:
# ============================================================
# 節1: データを読み込んで「どんなデータか」をつかむ（EDA）
#   狙い = 節2の前処理を決めるための材料集め
#   見るのは4点: 形 / 中身 / 欠損 / 目的変数の偏り
# ============================================================
import pandas as pd

# train = 答え(Survived)つきの学習用 / test = 答えなし・これを予測して提出する
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

# 1) 形: 行数(人数)と列数。test は Survived が無いぶん1列少ない = これが本番の構造
print("train:", train.shape, "  test:", test.shape)

# 2) 中身: 実際の値を数行眺める（どんな列があるか掴む）
display(train.head())

# 3) 欠損: 列ごとの「空っぽ」の数を多い順に ← 節2で「埋める/捨てる」を決める根拠
print("--- train 欠損数（多い順）---")
print(train.isnull().sum().sort_values(ascending=False))
print("\n--- test 欠損数（多い順）---")
print(test.isnull().sum().sort_values(ascending=False))

# 4) 目的変数の偏り: Survived の 0/1 割合
#    「全員を多数派(0)と予測」しただけで取れる点 = 何もしないベースラインの感覚
print("\n--- Survived の割合（0=死亡 / 1=生存）---")
print(train["Survived"].value_counts(normalize=True).round(3))

train: (891, 12)   test: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


--- train 欠損数（多い順）---
Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Fare             0
dtype: int64

--- test 欠損数（多い順）---
Cabin          327
Age             86
Fare             1
PassengerId      0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Embarked         0
dtype: int64

--- Survived の割合（0=死亡 / 1=生存）---
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64


## 2. 前処理

In [2]:
# ============================================================
# 節2: 前処理 + 特徴量づくり
#   なぜ必要か:
#    - モデルは「欠損」と「文字」を食べられない → 埋める & 数値化する
#    - 生の列だけより、組み合わせた特徴量のほうが生死のパターンを掴みやすい
# ============================================================

# --- 補完の基準値は train だけから作る ---
# test の統計で埋めると「テストの答えを覗き見」になる(リーク) → train の値で統一する
age_fill  = train["Age"].median()       # Age: 2割欠損 → 中央値で埋める
fare_fill = train["Fare"].median()      # Fare: test に1件だけ欠損
emb_fill  = train["Embarked"].mode()[0] # Embarked: train に2件だけ欠損 → 最頻値

def preprocess(df):
    """train/test に全く同じ処理をかけるための関数（処理がズレると予測が壊れる）"""
    df = df.copy()

    # 1) 欠損を埋める（Cabin は8割欠損なので今回は列ごと使わない）
    df["Age"]      = df["Age"].fillna(age_fill)
    df["Fare"]     = df["Fare"].fillna(fare_fill)
    df["Embarked"] = df["Embarked"].fillna(emb_fill)

    # 2) 文字 → 数値（モデルは数値しか食べないので変換）
    df["Sex"]      = df["Sex"].map({"male": 0, "female": 1})
    df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

    # 3) 新しい特徴量を作る（生の列の「裏にある意味」を取り出す）
    # 家族の人数: 兄弟配偶者(SibSp) + 親子(Parch) + 本人
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    # 単身フラグ: 一人客は生存率が違う、という仮説
    df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)
    # 敬称: 名前 "Braund, Mr. Owen" の Mr/Mrs/Miss/Master を抜き出す
    #   → 性別・年齢・既婚っぽさが詰まった強い手がかり。表に無いものは 4 にまとめる
    df["Title"]      = df["Name"].str.extract(r",\s*([^.]+)\.")
    df["Title"]      = df["Title"].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3}).fillna(4).astype(int)
    return df

train_p = preprocess(train)
test_p  = preprocess(test)

# モデルに渡す列(特徴量)を確定。X=入力 / y=正解 / X_test=予測したい相手
features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize", "IsAlone", "Title"]
X      = train_p[features]
y      = train_p["Survived"]
X_test = test_p[features]

print("特徴量:", features)
print("X:", X.shape, " X_test:", X_test.shape)
# 欠損残りが 0 なら前処理が成功している（モデルに渡せる状態）
print("欠損残り  X:", int(X.isnull().sum().sum()), " / X_test:", int(X_test.isnull().sum().sum()))
display(X.head())

特徴量: ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'Title']
X: (891, 8)  X_test: (418, 8)
欠損残り  X: 0  / X_test: 0


,Pclass,Sex,Age,Fare,Embarked,FamilySize,IsAlone,Title
0,3,0,22.0,7.2500,0,2,0,0
1,1,1,38.0,71.2833,1,2,0,2
2,3,1,26.0,7.9250,0,1,1,1
3,1,1,35.0,53.1000,0,2,0,2
4,3,0,35.0,8.0500,0,1,1,0


## 3. モデル ＋ 手元CV　｜　✋ まず予想：手元CVは何点？ なぜ？

In [3]:
# ============================================================
# 節3: モデルを作って「手元で」実力を測る（CV = クロスバリデーション）
#   なぜCVか: test の答えは見られない。だから train を分割して
#            「学習に使ってない部分」で自己採点し、未知データへの強さを推定する
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, StratifiedKFold

# LightGBM = 決定木を少しずつ継ぎ足して賢くする勾配ブースティング(GBDT)
model = lgb.LGBMClassifier(
    n_estimators=300,     # 継ぎ足す木の本数
    learning_rate=0.05,   # 1本ごとの学習の歩幅（小さい=慎重）
    num_leaves=15,        # 木の複雑さ。小さめ=過学習を抑える(Titanicは891行と小さい)
    subsample=0.8,        # 各木は行の8割だけ使う（過学習よけ）
    colsample_bytree=0.8, # 各木は列の8割だけ使う（過学習よけ）
    random_state=42,      # 乱数固定=毎回同じ結果（再現性）
    verbose=-1,           # 学習ログを黙らせる
)

# 5分割。層化(Stratified)=各分割で Survived の 0/1 比率を元データと同じに保つ（偏り防止）
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 「4/5で学習 → 残り1/5で採点」を5回繰り返し、5つのスコアを返す
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("fold ごとの点:", scores.round(4))
print("手元CV accuracy: %.4f ± %.4f" % (scores.mean(), scores.std()))  # 平均 ± ブレ幅

fold ごとの点: [0.8492 0.8427 0.8202 0.8596 0.8596]
手元CV accuracy: 0.8462 ± 0.0145


## 4. test を予測 → submission.csv（PassengerId, Survived / index=False）

In [4]:
# ============================================================
# 節4: test を予測して提出ファイルを作る
#   CVは「採点」だけで本物のモデルは残らない → ここで全 train で学習し直す
# ============================================================
model.fit(X, y)               # train 全部(891人)で学習 = CVより多いデータで本番モデルを作る
pred = model.predict(X_test)  # 答えを知らない test 418人の生死を予測

# 提出フォーマット: PassengerId と Survived の2列だけ
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": pred.astype(int),   # 0/1 の整数で
})
# index=False = 行番号を書かない（Kaggleの提出形式に合わせる）
submission.to_csv("submissions/submission.csv", index=False)

print("submission:", submission.shape)
print(submission["Survived"].value_counts())  # 何人を生存/死亡と予測したか
display(submission.head())

submission: (418, 2)
Survived
0    267
1    151
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## 5. 提出（自分で）

```
kaggle competitions submit -c titanic -f submissions/submission.csv -m "メッセージ"
```
提出したら public スコアを持ってきて。予想と照らして「なぜその点か」を一緒に開ける。